# Streaming answers

`query` blocks until the whole answer exists. `stream_query` yields
`SearchEngineStreamEvent` objects as the model produces text, which is what an
interactive UI needs — first-token latency drops from "as long as the answer takes"
to "as long as retrieval takes".

Three things about the contract are worth knowing:

- Retrieval is **not** streamed. It runs to completion first, then generation
  streams. Every event therefore carries the same fully-populated `retrieval`, so a
  consumer processing one chunk at a time can still cite sources without waiting
  for the end.
- Streaming is **text-only**. A Pydantic schema cannot be validated before the
  response is complete, so structured output has to go through `query`.
- `QueryPlanEngine` streams only the final subquery. Everything the plan depends on
  is answered normally first, because those answers are inputs to later prompts
  rather than output for the user.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL`.

In [ ]:
import os
from collections.abc import AsyncIterator
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    GlobalSearchEngine,
    KnowledgeGraph,
    LocalSearchEngine,
    MixSearchEngine,
    NaiveSearchEngine,
    QueryPlanEngine,
    SearchEngineStreamEvent,
    Settings,
    SimpleChunker,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.search_engine.local_search import LocalParams
from ragu.search_engine.naive_search import NaiveSearchParams
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")
QUESTION = "Where did the father of the creator of the C programming language work?"


async def consume(events: AsyncIterator[SearchEngineStreamEvent]) -> None:
    """
    Print a stream as it arrives, then report what the events carried.

    :param events: Event stream returned by ``stream_query``.
    """
    count = 0
    retrieval = None
    async for event in events:
        print(event.delta, end="", flush=True)
        # The retrieval container is the same object on every event.
        retrieval = event.retrieval
        count += 1

    print(f"\n\n[events: {count}, retrieval: {type(retrieval).__name__}]")

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/streaming_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

## Build the graph

The expensive cell. `make_community_summary=True` is what `GlobalSearchEngine`
needs further down.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(make_community_summary=True),
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))

In [ ]:
local = LocalSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
naive = NaiveSearchEngine(llm=llm, knowledge_graph=knowledge_graph, embedder=embedder)
global_search = GlobalSearchEngine(llm=llm, knowledge_graph=knowledge_graph)
mix = MixSearchEngine(llm=llm, engines=[local, naive])
planner = QueryPlanEngine(local)

local_params = LocalParams(top_k=10)

## Local search

Watch where the pause is: nothing prints until retrieval finishes, then text
arrives continuously.

In [ ]:
await consume(local.stream_query(QUESTION, local_params))

## Naive search

Shortest retrieval of the five, so the first token lands soonest.

In [ ]:
await consume(naive.stream_query(QUESTION, NaiveSearchParams(top_k=10)))

## Global search

The longest pause before the first token: it rates every community summary before
it can start writing.

In [ ]:
await consume(global_search.stream_query(QUESTION))

## Mix search

Both children retrieve first, then one synthesis streams.

In [ ]:
await consume(mix.stream_query(QUESTION))

## Query planning

Only the final subquery streams; the dependency hops are answered silently first.

In [ ]:
await consume(planner.stream_query(QUESTION, local_params))

In [ ]:
await knowledge_graph.index.close()